## Set up Data

#### Mount drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#### Unzip the Images

In [ ]:
import zipfile, os

CHALLENGE_DIR = "/content/drive/MyDrive/AML"
ZIP_PATH      = os.path.join(CHALLENGE_DIR, "dataset.zip")
IMAGES_DIR    = "/content/images"   # extracted here (fast local SSD, not Drive)

if not os.path.exists(IMAGES_DIR):
    print("Extracting images...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(IMAGES_DIR)
    print(f"Done — extracted to {IMAGES_DIR}")
else:
    print(f"Already extracted: {IMAGES_DIR}")

# Peek at what's inside
sample_files = os.listdir(os.path.join(IMAGES_DIR,"images"))[:5]
print(f"Total images : {len(os.listdir(os.path.join(IMAGES_DIR,"images")))}")
print(f"Sample names : {sample_files}")
IMAGES_DIR = os.path.join(IMAGES_DIR,"images")

#### Load the Training Data

In [ ]:
import pandas as pd

train_df = pd.read_csv(os.path.join(CHALLENGE_DIR, "train.csv"))
test_df  = pd.read_csv(os.path.join(CHALLENGE_DIR, "test.csv"))

# Map image_id → full path
def img_path(image_id):
    # Try common extensions; adjust if your images have a fixed extension
    for ext in ["", ".jpg", ".jpeg", ".png", ".webp"]:
        p = os.path.join(IMAGES_DIR, f"{image_id}{ext}")
        if os.path.exists(p):
            return p
    return None

train_df["path"] = train_df["image_id"].apply(img_path)
test_df["path"]  = test_df["image_id"].apply(img_path)

print(f"Train rows   : {len(train_df)}")
print(f"Test  rows   : {len(test_df)}")
print("\nLabel distribution:")
print(train_df["ground_truth"].value_counts().rename({0: "authentic", 1: "ai_generated"}))


## EfficientNetV2 Fine Tuning

In [ ]:
!pip install timm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from sklearn.metrics import f1_score

class AIDetectionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx]['path']
        label = self.dataframe.iloc[idx]['ground_truth']
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

FOLD_TO_RUN = 4
TOTAL_FOLDS = 10

skf = StratifiedKFold(n_splits=TOTAL_FOLDS, shuffle=True, random_state=42)

folds = list(skf.split(train_df, train_df["ground_truth"]))
train_val_idx, test_idx = folds[FOLD_TO_RUN]


test_data = train_df.iloc[test_idx]
pool_data = train_df.iloc[train_val_idx]
train_data, val_data = train_test_split(
    pool_data,
    test_size=0.111,
    stratify=pool_data["ground_truth"],
    random_state=42
)

print(f"EXPERIMENT RUN: Fold {FOLD_TO_RUN}")
print(f"Training samples:   {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Test samples:       {len(test_data)}")

# train_val_data, test_data = train_test_split(train_df, test_size=0.1, stratify=train_df["ground_truth"], random_state=42)
# train_data, val_data = train_test_split(train_val_data, test_size=0.111, stratify=train_val_data["ground_truth"], random_state=42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "tf_efficientnetv2_s.in21k"
model_path = os.path.join(CHALLENGE_DIR, f"models/{model_name}_try{FOLD_TO_RUN + 1}.pth")

### Basic

In [ ]:
import timm

model = timm.create_model(model_name, pretrained=True, num_classes=2, drop_rate=0.3, drop_path_rate=0.2)
model = model.to(device)

data_config = timm.data.resolve_model_data_config(model)
train_transform = timm.data.create_transform(**data_config, is_training=True)
val_transform = timm.data.create_transform(**data_config, is_training=False)

train_dataset = AIDetectionDataset(train_data, transform=train_transform)
val_dataset = AIDetectionDataset(val_data, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

### With data augmentation

In [ ]:
import timm

model = timm.create_model(model_name, pretrained=True, num_classes=2, drop_rate=0.3, drop_path_rate=0.2)
model = model.to(device)

In [ ]:
import torchvision.transforms.v2 as T

data_config = timm.data.resolve_model_data_config(model)
input_size = data_config['input_size'][-2:]
mean = data_config['mean']
std = data_config['std']

train_transform = T.Compose([
    T.RandomResizedCrop(input_size, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    T.RandomApply([T.GaussianBlur(kernel_size=5, sigma=(0.1, 1.0))], p=0.3),
    T.RandomApply([T.JPEG((60, 95))], p=0.5),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=mean, std=std),
])

val_transform = T.Compose([
    T.Resize(input_size),
    T.CenterCrop(input_size),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=mean, std=std),
])

train_dataset = AIDetectionDataset(train_data, transform=train_transform)
val_dataset = AIDetectionDataset(val_data, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

### Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-2)
epochs = 25
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

best_f1 = 0.0

for epoch in range(epochs):
    model.train()
    train_loss = 0
    train_preds = []
    train_labels = []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs = model(images)
            loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        _, preds = torch.max(outputs, 1)
        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

        pbar.set_postfix({'lr': f"{scheduler.get_last_lr()[0]:.2e}"})

    avg_train_loss = train_loss / len(train_loader)
    train_f1 = f1_score(train_labels, train_preds)

    model.eval()
    val_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            v_loss = criterion(outputs, labels)
            val_loss += v_loss.item()

            _, preds = torch.max(outputs, 1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_f1 = f1_score(val_labels, val_preds)

    print(f"Epoch {epoch+1} Results:")
    print(f"  Train -> Loss: {avg_train_loss:.4f} | F1: {train_f1:.4f}")
    print(f"  Val   -> Loss: {avg_val_loss:.4f} | F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), os.path.join(CHALLENGE_DIR, model_path))
        print(f"--> New best model saved")

    scheduler.step()